In [36]:
from pathlib import Path
import shutil
import yaml
import cv2
import xml.etree.ElementTree as ET

print("Libraries loaded")

Libraries loaded


In [38]:
from pathlib import Path


PROJECT_ROOT = Path(
    r"C:\Users\leewa\Downloads\PCB-Defect-Inspection"
)


# Processed images from pipeline
PROCESSED_ROOT = (
    PROJECT_ROOT
    /
    "data"
    /
    "processed"
    /
    "leewanching"
)


# Original XML annotations
ANNOTATION_ROOT = (
    PROJECT_ROOT
    /
    "data"
    /
    "raw"
    /
    "Annotations"
)


SPLIT_ROOT = (
    PROJECT_ROOT
    /
    "splits"
)


print(PROCESSED_ROOT)
print(ANNOTATION_ROOT)

C:\Users\leewa\Downloads\PCB-Defect-Inspection\data\processed\leewanching
C:\Users\leewa\Downloads\PCB-Defect-Inspection\data\raw\Annotations


In [39]:
print(PROCESSED_ROOT.exists())
print(ANNOTATION_ROOT.exists())

True
True


In [40]:
DATASETS = [
    "set1",
    "set2",
    "set3"
]


SPLITS = [
    "train",
    "val",
    "test"
]


print(DATASETS)

['set1', 'set2', 'set3']


In [41]:
for dataset in DATASETS:

    for split in SPLITS:

        label_folder = (
            PROCESSED_ROOT
            /
            dataset
            /
            "labels"
            /
            split
        )


        label_folder.mkdir(
            parents=True,
            exist_ok=True
        )


print("Label folders created")

Label folders created


In [42]:
CLASS_NAMES = {

    "missing_hole": 0,

    "mouse_bite": 1,

    "open_circuit": 2,

    "short": 3,

    "spur": 4,

    "spurious_copper": 5

}


print(CLASS_NAMES)

{'missing_hole': 0, 'mouse_bite': 1, 'open_circuit': 2, 'short': 3, 'spur': 4, 'spurious_copper': 5}


In [43]:
def convert_xml_to_yolo(
    xml_file,
    txt_file,
    img_width,
    img_height
):

    tree = ET.parse(
        xml_file
    )

    root = tree.getroot()


    yolo_labels = []


    for obj in root.findall("object"):


        class_name = (
            obj.find("name")
            .text
            .strip()
        )


        if class_name not in CLASS_NAMES:

            continue


        class_id = CLASS_NAMES[class_name]


        bbox = obj.find(
            "bndbox"
        )


        xmin = float(
            bbox.find("xmin").text
        )

        ymin = float(
            bbox.find("ymin").text
        )

        xmax = float(
            bbox.find("xmax").text
        )

        ymax = float(
            bbox.find("ymax").text
        )


        x_center = (
            (xmin + xmax) / 2
        ) / img_width


        y_center = (
            (ymin + ymax) / 2
        ) / img_height


        width = (
            xmax - xmin
        ) / img_width


        height = (
            ymax - ymin
        ) / img_height


        yolo_labels.append(
            f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
        )


    with open(
        txt_file,
        "w"
    ) as f:

        f.write(
            "\n".join(yolo_labels)
        )

In [44]:
for dataset in DATASETS:

    print("\nPreparing:", dataset)


    for split in SPLITS:


        image_folder = (
            PROCESSED_ROOT
            /
            dataset
            /
            "images"
            /
            split
        )


        label_folder = (
            PROCESSED_ROOT
            /
            dataset
            /
            "labels"
            /
            split
        )


        # search images inside class folders
        image_files = list(
            image_folder.rglob("*.jpg")
        )


        print(
            split,
            "images:",
            len(image_files)
        )


        for image_file in image_files:


            # search XML anywhere in Annotations
            xml_files = list(
                ANNOTATION_ROOT.rglob(
                    f"{image_file.stem}.xml"
                )
            )


            if len(xml_files) == 0:

                print(
                    "Missing XML:",
                    image_file.name
                )

                continue


            xml_file = xml_files[0]


            image = cv2.imread(
                str(image_file)
            )


            if image is None:

                continue


            height, width = image.shape[:2]


            label_file = (
                label_folder
                /
                f"{image_file.stem}.txt"
            )


            convert_xml_to_yolo(
                xml_file,
                label_file,
                width,
                height
            )


print("\nAnnotation conversion completed")


Preparing: set1
train images: 481
val images: 60
test images: 152

Preparing: set2
train images: 481
val images: 60
test images: 152

Preparing: set3
train images: 481
val images: 60
test images: 152

Annotation conversion completed


In [45]:
CLASS_YAML = {

    0: "missing_hole",

    1: "mouse_bite",

    2: "open_circuit",

    3: "short",

    4: "spur",

    5: "spurious_copper"

}


for dataset in DATASETS:


    yaml_file = (
        PROCESSED_ROOT
        /
        dataset
        /
        f"{dataset}.yaml"
    )


    yaml_content = {


        "path":
        str(
            (
                PROCESSED_ROOT
                /
                dataset
            ).resolve()
        ),


        "train":
        "images/train",


        "val":
        "images/val",


        "test":
        "images/test",


        "names":
        CLASS_YAML

    }


    with open(
        yaml_file,
        "w"
    ) as f:

        yaml.safe_dump(
            yaml_content,
            f,
            sort_keys=False
        )


    print(
        "Created:",
        yaml_file
    )

Created: C:\Users\leewa\Downloads\PCB-Defect-Inspection\data\processed\leewanching\set1\set1.yaml
Created: C:\Users\leewa\Downloads\PCB-Defect-Inspection\data\processed\leewanching\set2\set2.yaml
Created: C:\Users\leewa\Downloads\PCB-Defect-Inspection\data\processed\leewanching\set3\set3.yaml


In [46]:
for dataset in DATASETS:

    print("\n================")
    print(dataset)


    images = list(
        (
            PROCESSED_ROOT
            /
            dataset
            /
            "images"
            /
            "train"
        ).rglob("*.jpg")
    )


    labels = list(
        (
            PROCESSED_ROOT
            /
            dataset
            /
            "labels"
            /
            "train"
        ).glob("*.txt")
    )


    print(
        "Train images:",
        len(images)
    )


    print(
        "Train labels:",
        len(labels)
    )


set1
Train images: 481
Train labels: 481

set2
Train images: 481
Train labels: 481

set3
Train images: 481
Train labels: 481
